# Class 08 · Reproducible Reporting (R)

Live-coding companion to the Week 4, Class B slides. Run the cells from top to bottom.
We build one Iris report step by step, render it from a fresh R process, then add a parameter
to a second report and watch coverage change.

Keep this notebook in the folder that also holds `class-08-iris.Rmd`, `class-08-steam.Rmd` and
`steam_achievements.csv`.

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 3)

## Part 1 · Why a dynamic document

- A result copied by hand into a report goes stale when the data change.
- A dynamic document keeps the question, code, evidence and interpretation in one source.
- Computed text updates a count after a rerun. An interpretation sentence still needs to be checked.
- A PDF records one run. The source still needs its input data and software to reproduce that run.

**Two separate stages.** Execution runs the code (R and knitr). Conversion turns the result into a
document (Pandoc, then XeLaTeX for PDF). The failing stage tells you which log to read.

## Part 2 · Build the Iris report, one chunk at a time

**Question.** How do sepal lengths vary within and between the three species? Iris has 150 flowers, 50 per species, measured in centimetres.

### Document settings

The YAML header names the output formats; the first is the default.

In [ ]:
cat(readLines("class-08-iris.Rmd", n = 10), sep = "\n")

### Setup (in the report: `include=FALSE` runs it but hides code and output)

In [ ]:
library(ggplot2)
library(dplyr)
# In the .Rmd, this also sets chunk options:
# knitr::opts_chunk$set(
#   echo = FALSE, cache = FALSE, error = FALSE,
#   warning = TRUE, message = FALSE,
#   fig.width = 6, fig.height = 3, out.width = "90%"
# )
dat <- iris

### Computed prose

In the `.Rmd`, an inline `` `r ...` `` expression writes the value into the sentence.

In [ ]:
cat("The mean sepal length is", sprintf("%.2f", mean(dat$Sepal.Length)), "cm,",
    "with sample standard deviation", sprintf("%.3f", sd(dat$Sepal.Length)), "cm.\n")

> Check: mean 5.84 cm, sample SD 0.828 cm.

### A compact group summary

`.groups = "drop"` returns an ordinary table; `kable` formats it for the report.

In [ ]:
summary_table <- dat |>
  group_by(Species) |>
  summarise(
    n = n(), Sepal_cm = mean(Sepal.Length),
    Petal_cm = mean(Petal.Length), .groups = "drop"
  )
knitr::kable(summary_table, row.names = FALSE, digits = 3)

> Every species has 50 flowers. Means alone hide the spread, so the report also needs plots.

### Figure 1: the pooled distribution

Record the bin grid in the report: 0.4 cm bins from 4.25 to 8.25 cm.

In [ ]:
edges <- seq(4.25, 8.25, length.out = 11)
ggplot(dat, aes(Sepal.Length)) +
  geom_histogram(
    breaks = edges, closed = "left",
    fill = "#D4A017", colour = "white"
  ) +
  labs(x = "Sepal length (cm)", y = "Number of flowers") +
  theme_minimal(base_size = 11)

In [ ]:
hist(dat$Sepal.Length, breaks = edges, right = FALSE, plot = FALSE)$counts  # 9 23 20 28 19 23 16 6 5 1

### Figure 2: compare the species

Linear quartiles and 1.5-IQR whiskers. In the `.Rmd`, `fig.width`/`fig.height` are inches, `out.width` is the display width, `fig.cap` the caption.

In [ ]:
ggplot(dat, aes(Species, Sepal.Length)) +
  geom_boxplot(fill = "#D4A017", width = 0.55) +
  labs(x = "Species", y = "Sepal length (cm)") +
  theme_minimal(base_size = 11)

> Setosa has the lowest median; the ranges overlap. The separate virginica point is 4.9 cm.

### Reproduction record

In [ ]:
cat("R", as.character(getRversion()), "\n")
for (pkg in c("ggplot2", "dplyr", "knitr", "rmarkdown")) {
  cat(pkg, as.character(packageVersion(pkg)), "\n")
}

## Part 3 · Render from a fresh R process

`Rscript --vanilla` starts a clean session, so the PDF cannot depend on objects left in this notebook. knitr executes the chunks; Pandoc and XeLaTeX build the PDF.

In [ ]:
render <- function(source, ...) {
  call <- sprintf("rmarkdown::render(%s, output_format = 'pdf_document', quiet = TRUE%s)",
                  deparse(source), paste0(", ", c(...), collapse = ""))
  if (length(c(...)) == 0) call <- sub(", $", "", call)
  status <- system2("Rscript", c("--vanilla", "-e", shQuote(call)))
  cat("exit status", status, "\n")
}

render("class-08-iris.Rmd")
file.exists("class-08-iris.pdf")

## Part 4 · Check the result that will be submitted

1. Confirm the command completed without an execution error.
2. Open the new PDF and inspect every page.
3. Verify one count and one calculated value against the input.
4. Check every plot, unit, table and interpretation.
5. Preserve the source, the input identity and the software versions.

A successful render proves this configured run completed. It does not prove that the analysis
or the conclusion is correct.

## Part 5 · A parameterized report

A **parameter** names an input choice and records it. Changing it changes the selection while
the analytical recipe stays the same.

Data: six game records. A row is one game.

$$\text{completion rate} = \frac{\text{completed players}}{\text{listed players}}$$

- Completed players cannot exceed listed players.
- Players must be positive for a defined rate.
- The achievement count is **not** the rate denominator.

### Declare defaults in the YAML header

In [ ]:
cat(readLines("class-08-steam.Rmd", n = 13), sep = "\n")

Inside the report, `params$minplayers` reads the value. Here we create `params` by hand.

In [ ]:
params <- list(datafile = "steam_achievements.csv", minplayers = 100)
library(ggplot2)

### Validate, filter a copy, compute the rate

`raw` is kept unchanged; every assumption about the counts is checked.

In [ ]:
raw <- read.csv(params$datafile, stringsAsFactors = FALSE)
required <- c("game", "players", "completed", "achievements")
stopifnot(all(required %in% names(raw)), !anyNA(raw[required]))
counts <- raw[c("players", "completed", "achievements")]
stopifnot(all(vapply(counts, is.numeric, logical(1))))
stopifnot(all(is.finite(as.matrix(counts))))
stopifnot(all(counts >= 0), all(counts == floor(as.matrix(counts))))
stopifnot(all(raw$players > 0), all(raw$completed <= raw$players))
stopifnot(!anyDuplicated(raw$game), all(nzchar(raw$game)))
stopifnot(length(params$minplayers) == 1,
          is.finite(params$minplayers), params$minplayers >= 0)
cleaned <- raw[raw$players >= params$minplayers, ]
cleaned$rate <- cleaned$completed / cleaned$players
cleaned <- cleaned[order(-cleaned$rate, cleaned$game), ]
cat("The threshold is at least", params$minplayers, "players per game. This retains",
    nrow(cleaned), "of", nrow(raw), "games.\n")

### Table and two figures

In [ ]:
shown <- cleaned[c("game", "players", "completed", "achievements", "rate")]
knitr::kable(shown, row.names = FALSE, digits = 3)

In [ ]:
if (nrow(cleaned) == 0) {
  cat("No games meet this threshold. There are no rates to plot.\n")
} else {
  cleaned$game <- factor(cleaned$game, levels = rev(cleaned$game))
  print(ggplot(cleaned, aes(rate, game)) +
    geom_col(fill = "#D4A017") +
    scale_x_continuous(limits = c(0, 1)) +
    labs(x = "Completed / players", y = "Game") +
    theme_minimal(base_size = 11))
}

In [ ]:
if (nrow(cleaned) > 0) {
  print(ggplot(cleaned, aes(players, game)) +
    geom_col(fill = "#777777") +
    labs(x = "Number of players", y = "Game") +
    theme_minimal(base_size = 11))
}

> At 100 players, four of six games remain. The rate plot compares proportions; the count plot shows how many players each proportion describes.

### Change the parameter

Predict first: how many games remain at 300? At 1000?

In [ ]:
for (threshold in c(100, 300, 1000)) {
  kept <- raw[raw$players >= threshold, ]
  cat(threshold, ":", nrow(kept), "of", nrow(raw), "games;",
      round(kept$completed / kept$players, 3), "\n")
}

> 300 keeps three games with the **same** per-game rates; filtering changes coverage, not the rate. 1000 keeps none: say so plainly instead of drawing an empty chart.

### Render one PDF per parameter value

In [ ]:
render("class-08-steam.Rmd", "params = list(minplayers = 300)", "output_file = 'report-300.pdf'")
file.exists("report-300.pdf")

## Part 6 · Troubleshooting and what to preserve

- Missing name or package: check setup and the selected environment.
- Missing file: check the working folder and the relative path.
- Pandoc error: check document conversion. XeLaTeX error: check the PDF compilation log.
- Preserve the source, raw input identity, parameters, software versions and the PDF from the final fresh run.